<a href="https://colab.research.google.com/github/NicolasGarridoB/actividad-1.4.2-programacion-cd/blob/main/Mi_flujo_profesional_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset Titanic para actividad 1.4.2


In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('titanic_data.csv')
print('----------------------------------------------------------------------')
print(df.info())
print('----------------------------------------------------------------------')
print(df.describe())
print('----------------------------------------------------------------------')
print(f"\nValores nulos por columna:\n{df.isnull().sum()}")

----------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 889 entries, 0 to 888
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     889 non-null    int64  
 1   pclass       889 non-null    int64  
 2   sex          889 non-null    object 
 3   age          713 non-null    float64
 4   sibsp        889 non-null    int64  
 5   parch        889 non-null    int64  
 6   fare         889 non-null    float64
 7   embarked     887 non-null    object 
 8   class        889 non-null    object 
 9   who          889 non-null    object 
 10  adult_male   889 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  887 non-null    object 
 13  alive        889 non-null    object 
 14  alone        889 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.2+ KB
None
---------------------------------

In [36]:
# Eliminar duplicados
df_clean = df.drop_duplicates().copy()
# Imputar valores faltantes con la mediana
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
# Imputar Embarked con la moda
df_clean['embarked'] = df_clean['embarked'].fillna(df['embarked'].mode()[0])
df_clean['embark_town'] = df_clean['embark_town'].fillna(df['embark_town'].mode()[0])
df_clean


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,NaN,Queenstown,no,False
885,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
886,0,3,female,28.0,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
887,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [37]:
# Eliminar columna con demasiados nulos
df_clean.drop(columns=['deck'], inplace=True)

In [38]:
df_clean

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,Queenstown,no,False
885,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True
886,0,3,female,28.0,1,2,23.4500,S,Third,woman,False,Southampton,no,False
887,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True


In [40]:
# Manejo de Outliers en la columna 'fare'

# Rango Intercuartílico (IQR)
Q1 = df_clean['fare'].quantile(0.25)
Q3 = df_clean['fare'].quantile(0.75)
IQR = Q3 - Q1

# Definimos los límites (estándar de 1.5 * IQR)
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR


df_clean['fare'] = np.where(df_clean['fare'] > limite_superior, limite_superior, df_clean['fare'])
df_clean['fare'] = np.where(df_clean['fare'] < limite_inferior, limite_inferior, df_clean['fare'])

print(f"Límite superior definido: {limite_superior}")
print(df_clean[['fare']].describe())

Límite superior definido: 73.641125
             fare
count  782.000000
mean    26.597380
std     22.888455
min      0.000000
25%      8.050000
50%     15.950000
75%     34.286450
max     73.641125


In [49]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


pipeline = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first'), ['sex']),
    ('num', StandardScaler(), ['age', 'fare'])
])

X_transf = pipeline.fit_transform(df_clean)

In [50]:
df_clean

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,0,3,female,39.0,0,5,29.1250,Q,Third,woman,False,Queenstown,no,False
885,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True
886,0,3,female,28.0,1,2,23.4500,S,Third,woman,False,Southampton,no,False
887,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True


In [55]:
print('====================================================================================')
print(df_clean.info())
print('====================================================================================')
print(df_clean.describe())

print('====================================================================================')
print(df.info())
print('====================================================================================')
print(df.describe())


<class 'pandas.core.frame.DataFrame'>
Index: 782 entries, 0 to 888
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     782 non-null    int64  
 1   pclass       782 non-null    int64  
 2   sex          782 non-null    object 
 3   age          782 non-null    float64
 4   sibsp        782 non-null    int64  
 5   parch        782 non-null    int64  
 6   fare         782 non-null    float64
 7   embarked     782 non-null    object 
 8   class        782 non-null    object 
 9   who          782 non-null    object 
 10  adult_male   782 non-null    bool   
 11  embark_town  782 non-null    object 
 12  alive        782 non-null    object 
 13  alone        782 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(6)
memory usage: 80.9+ KB
None
         survived      pclass         age       sibsp       parch        fare
count  782.000000  782.000000  782.000000  782.000000  782.000000  782.00

**Resumen:**


# Estado dataset inicial:

**Registros iniciales**: 891 filas

**Registros nulos**: Edad(177), Embarked(2)

**Columna 'Deck'**: 688 nulos.

**Outliers (Fare)**: Máximo: $512.32

**Variable 'Sex'**: "male", "female"

**Escala de Edad**: 0.42 a 80.0 años

# Estado dataset_clean:

**Registros iniciales**: 784 filas (Se eliminaron 107 duplicados para evitar sesgos.)

**Registros nulos**: 0 (Datos completos; se usó mediana y moda.)

**Columna 'Deck'**: Eliminada.

**Outliers (Fare)**: Máximo: $65.63 (Se limitaron valores extremos (Capping) con IQR.)

**Variable 'Sex'**: 1, 0 (Encoding: Texto convertido a binario para cálculo.)


**Escala de Edad**: -2.25 a 3.01 (StandardScaler para centrar los datos en la media).

El valor 0 representa la edad promedio de los pasajeros del Titanic (29-30 años).

Valores positivos (hasta 3.01): Son personas más viejas que el promedio.

Valores negativos (hasta -2.25): Son personas más jóvenes que el promedio.